In [1]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch
from pathlib import Path
import random
from tqdm.auto import tqdm
import json

In [2]:
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [3]:
PROMPT = """
You are a professional fashion annotator.

Describe ONLY the garment.

Include:

- garment category
- colors
- material if visible
- sleeve length
- neckline or collar
- fit
- fabric pattern
- graphic prints
- logos
- embroidery
- text on the garment
- decorations
- closures
- fashion style

Do not omit visible branding, text, or graphic elements.

Return a single comma-separated description.
"""

In [4]:
img_path = Path("/teamspace/studios/this_studio/data/train/cloth")

In [6]:
def generate_caption(img_path):

    image = Image.open(img_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=64
    )

    caption = processor.batch_decode(
        generated_ids[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )[0]

    return caption

In [6]:
all_images = (
    list(img_path.glob("*.jpg"))
    + list(img_path.glob("*.png"))
)

output_file = "train_captions.jsonl"

with open(output_file, "w", encoding="utf-8") as f:

    for img_path in tqdm(all_images, desc="Generating captions"):

        try:
            caption = generate_caption(img_path)

            item = {
                "file_name": img_path.name,
                "raw_caption": caption
            }

            f.write(
                json.dumps(item, ensure_ascii=False)
                + "\n"
            )

            f.flush()

        except Exception as e:

            print(f"Failed: {img_path.name}")
            print(e)

Generating captions:   0%|          | 0/11647 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [4]:
test_cloth_path = Path("/teamspace/studios/this_studio/vitonhd/test/cloth")

In [7]:
all_images = (
    list(test_cloth_path.glob("*.jpg"))
    + list(test_cloth_path.glob("*.png"))
)

output_file = "test_captions.jsonl"

with open(output_file, "w", encoding="utf-8") as f:

    for img_path in tqdm(all_images, desc="Generating captions"):

        try:
            caption = generate_caption(img_path)

            item = {
                "file_name": img_path.name,
                "raw_caption": caption
            }

            f.write(
                json.dumps(item, ensure_ascii=False)
                + "\n"
            )

            f.flush()

        except Exception as e:

            print(f"Failed: {img_path.name}")
            print(e)

Generating captions:   0%|          | 0/2032 [00:00<?, ?it/s]